# Flujo completo DXF -> discretizacion -> calibracion opcional

Procesa el DXF de ejemplo, exporta CSV y muestra como activar calibracion por segmento.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "dynaengine").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_DIR = ROOT / "examples" / "data"
MATERIALS_PATH = DATA_DIR / "section_01_materials.json"
materials = json.loads(MATERIALS_PATH.read_text(encoding="utf-8"))
print(f"Proyecto: {ROOT}")
print(f"Materiales cargados: {len(materials)}")

import pandas as pd
from dynaengine import (
    CalibrationSettings,
    export_dataframe,
    process_dxf_folder,
)

Proyecto: C:\Users\joel.alarcon\Desktop\_code\prismo\external\DynaEngine
Materiales cargados: 6


In [2]:
results = process_dxf_folder(
    section_folder=DATA_DIR,
    x_positions_by_file={"section_01": {f"failure_{index}": [250.0, 480.0] for index in range(1, 8)}},
    materials=materials,
    target_frequency_hz=25,
    calibrate=False,
    failure_types_by_file={"section_01": {f"failure_{index}": "tipo_de_falla" for index in range(1, 8)}},
    small_area_scale=0.01,
)
print(f"Columnas procesadas: {len(results)}")
list(results)

Columnas procesadas: 2


['section_01-failure_3-x250p0', 'section_01-failure_7-x480p0']

In [3]:
summary = []
for name, processed in results.items():
    summary.append({
        "column_name": name,
        "raw_layers": len(processed.raw),
        "discretized_segments": len(processed.discretized),
        "max_depth_m": processed.raw["bottom_m"].max(),
        "min_frequency_hz": processed.discretized["natural_frequency_hz"].min(),
    })
summary_df = pd.DataFrame(summary)
summary_df

,column_name,raw_layers,discretized_segments,max_depth_m,min_frequency_hz
0,section_01-failure_3-x250p0,4,42,112.87278,25.328390
1,section_01-failure_7-x480p0,4,47,153.82703,25.000077


In [4]:
output_dir = ROOT / "examples" / "output"
output_dir.mkdir(parents=True, exist_ok=True)

for name, processed in results.items():
    export_dataframe(processed.raw, output_dir / f"{name}_raw.csv")
    export_dataframe(processed.discretized, output_dir / f"{name}_discretized.csv")
export_dataframe(summary_df, output_dir / "columns_summary.csv")
print(output_dir)

C:\Users\joel.alarcon\Desktop\_code\prismo\external\DynaEngine\examples\output


In [5]:
# Calibracion de todas las columnas correctas con una discretizacion ligera para ejemplo.
# Para produccion, sube target_frequency_hz y/o scipy_starts/scipy_maxiter.
settings = CalibrationSettings(
    optimizer="scipy",
    scipy_starts=1,
    scipy_maxiter=8,
    random_state=1,
)

calibrated_results = process_dxf_folder(
    section_folder=DATA_DIR,
    x_positions_by_file={"section_01": {f"failure_{index}": [250.0, 480.0] for index in range(1, 8)}},
    materials=materials,
    target_frequency_hz=2,
    calibrate=True,
    calibration_settings=settings,
    failure_types_by_file={"section_01": {f"failure_{index}": "tipo_de_falla" for index in range(1, 8)}},
    small_area_scale=0.01,
)

calibrated_values = pd.concat(
    [processed.calibrated.assign(correct_column=name) for name, processed in calibrated_results.items()],
    ignore_index=True,
)
calibrated_values[[
    "correct_column",
    "segment_id",
    "source_layer_id",
    "x_position_m",
    "failure_surface_id",
    "failure_surface_name",
    "material_name",
    "top_m",
    "bottom_m",
    "theta_1",
    "theta_2",
    "theta_3",
    "theta_4",
    "theta_5",
    "P1",
    "P2",
    "P3",
    "dmin",
    "gqh_score",
    "mrdf_score",
]]


,correct_column,segment_id,source_layer_id,x_position_m,failure_surface_id,failure_surface_name,material_name,top_m,bottom_m,theta_1,theta_2,theta_3,theta_4,theta_5,P1,P2,P3,dmin,gqh_score,mrdf_score
0,section_01-failure_3-x250p0,1,1,250.0,3,failure_3,Grava arcillosa,0.000000,31.585110,-1.172526,0.065445,7.440467,1.0,4.328080e-01,0.578368,0.391843,3.451044,0.957962,-0.014039,-3.832506e-02
1,section_01-failure_3-x250p0,2,1,250.0,3,failure_3,Grava arcillosa,31.585110,63.170220,0.712551,-0.463178,0.010000,1.0,1.345487e-09,0.387030,0.207029,2.869494,0.872754,-0.039947,-6.828932e-02
2,section_01-failure_3-x250p0,3,2,250.0,3,failure_3,Grava pobremente gradada,63.170220,80.721750,1.453080,-1.010625,0.010000,1.0,1.741039e-06,0.233462,0.050000,30.000000,0.791098,-0.030822,-3.987978e-01
3,section_01-failure_3-x250p0,4,3,250.0,3,failure_3,Grava arcillosa,80.721750,82.872980,0.429821,-0.423716,0.010032,1.0,5.407279e-10,0.553365,0.369294,1.882501,0.772378,-0.033048,-3.875245e-02
4,section_01-failure_3-x250p0,5,4,250.0,3,failure_3,Grava arenosa,82.872980,112.872780,0.910146,-0.442800,0.010000,1.0,7.180502e-09,0.999934,0.194833,0.500036,7.082448,-0.039846,-7.961473e-01
5,section_01-failure_7-x480p0,1,1,480.0,7,failure_7,Grava arcillosa,0.000000,26.913815,-0.772089,-1.670321,3.016279,1.0,3.511285e-01,0.576480,0.346828,2.638987,0.923574,-0.012789,-3.404013e-01
6,section_01-failure_7-x480p0,2,1,480.0,7,failure_7,Grava arcillosa,26.913815,53.827630,0.930782,-0.444093,0.010000,1.0,7.423405e-09,0.256477,0.203905,27.889526,1.153274,-0.051457,-4.510494e-01
7,section_01-failure_7-x480p0,3,2,480.0,7,failure_7,Grava arenosa,53.827630,61.536030,1.364179,-0.519896,4.948154,1.0,2.361019e-01,0.999934,0.110032,15.249431,7.983739,-0.092798,-8.805020e-01
8,section_01-failure_7-x480p0,4,3,480.0,7,failure_7,Estrato no identificado 1,61.536030,88.827230,-0.778881,-0.351850,0.010044,1.0,8.285113e-07,0.590212,0.201036,15.833698,0.696816,-0.008754,-3.559806e-02
9,section_01-failure_7-x480p0,5,4,480.0,7,failure_7,Grava pobremente gradada,88.827230,121.327130,1.026600,-0.470824,0.010000,1.0,7.053187e-06,0.500000,0.525000,15.250000,5.000000,-0.014568,-1.000000e+12
